In [40]:
################################################################################
# ProjectTwoDashboard.ipynb
#
# Description:
#     This Jupyter Notebook implements the final Project Two dashboard for
#     Grazioso Salvare. It builds upon the Module Six Milestone by creating a
#     complete interactive dashboard using Python Dash (JupyterDash), MongoDB,
#     and Plotly.
#
#     The dashboard retrieves data from the Austin Animal Center (AAC)
#     database using the AnimalShelter CRUD module. It includes:
#       - An interactive DataTable (read-all from MongoDB)
#       - User filter controls for rescue categories
#       - A Leaflet map showing the location of the selected animal
#       - A bar chart displaying top breeds based on selected filters
#
# Design Pattern:
#     - Model: MongoDB via AnimalShelter.py
#     - View: Dash visual components (DataTable, Map, Chart)
#     - Controller: Dash callback functions for filters and updates
#
# Dependencies:
#     pymongo, pandas, plotly, dash, dash-leaflet, jupyter-dash
#
# Author: Michael Langille
# Date: October 2025
# Course: CS-340 - Client/Server Development
# Southern New Hampshire University
#
# Notes:
#     - Ensure MongoDB is running before launching the dashboard.
#     - Place this notebook and AnimalShelter.py in the same directory.
#     - Store the logo under an 'assets' folder in the same directory.
#     - Use quiet mode (Cell 7) for clean screenshots.
################################################################################

# -------------------- Imports and App Setup --------------------
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

from dash import html, dcc, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl
import plotly.express as px
import pandas as pd
import base64
from pathlib import Path

# Import CRUD module
from AnimalShelter import AnimalShelter

# Initialize JupyterDash app and point to the 'assets' folder
app = JupyterDash("CS-340 Project Two — Grazioso Salvare", assets_folder="assets")

# Display logo from the assets folder using Dash's asset helper
logo = html.Img(src=app.get_asset_url("GraziosoSalvareLogo.png"),
                height="80px", style={"margin": "8px 0"})

# -------------------- MongoDB Configuration --------------------
USERNAME = "aacuser"
PASSWORD = "SNHU2025"   # <-- Update before running
HOST = "localhost"
PORT = 27017
AUTH_SOURCE = "admin"
DB_NAME = "aac"
COLLECTION = "animals"

# Connect to MongoDB via AnimalShelter class
shelter = AnimalShelter(
    USERNAME, PASSWORD,
    host=HOST, port=PORT, auth_source=AUTH_SOURCE,
    db_name=DB_NAME, collection_name=COLLECTION, tls=False
)

# Retrieve all records (unfiltered)
records = shelter.read({})
df = pd.DataFrame.from_records(records)

# Drop MongoDB _id field if present
if "_id" in df.columns:
    df = df.drop(columns=["_id"])

# Preserve a copy of full data
df_master = df.copy()

# Unique identifier for dashboard display
UNIQUE_ID = "ml-grazioso-dash"

# Verify data preview
(len(df), list(df.columns)[:12])


(10000,
 ['rec_num',
  'age_upon_outcome',
  'animal_id',
  'animal_type',
  'breed',
  'color',
  'date_of_birth',
  'datetime',
  'monthyear',
  'name',
  'outcome_subtype',
  'outcome_type'])

In [41]:
app = JupyterDash("CS-340 Project Two — Grazioso Salvare")

# Display logo (optional)
logo = html.Img(src="assets/Grazioso Salvare Logo.png", height="80px", style={"margin":"8px 0"})

app.layout = html.Div(
    style={"fontFamily": "sans-serif", "padding": "12px"},
    children=[
        html.Center(html.H1("Grazioso Salvare Dashboard — Project Two")),
        html.Center(logo),
        html.Center(html.Div(f"Unique ID: {UNIQUE_ID}", style={"marginBottom": "10px", "fontWeight":"bold"})),
        html.Hr(),

        html.H3("Select Rescue Type"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                {"label": "Reset (All)", "value": "reset"},
            ],
            value="reset",
            labelStyle={"display": "block", "margin": "4px 0"}
        ),

        html.Hr(),

        dash_table.DataTable(
            id="datatable-id",
            columns=[{"name": c, "id": c, "selectable": True} for c in df.columns],
            data=df.to_dict("records"),
            row_selectable="single",
            selected_rows=[0],
            filter_action="native",
            sort_action="native",
            sort_mode="multi",
            page_action="native",
            page_current=0,
            page_size=10,
            style_table={"overflowX": "auto"},
            style_header={"fontWeight": "bold"},
            style_cell={"minWidth": "120px", "whiteSpace": "normal", "height": "auto"},
        ),

        html.Br(),
        html.Div(
            style={"display": "flex", "gap": "16px", "flexWrap": "wrap"},
            children=[
                html.Div(id="map-id", className="col s12 m6"),
                html.Div(dcc.Graph(id="second-chart-id"), className="col s12 m6", style={"minWidth": "400px"})
            ]
        ),
    ],
)


In [42]:
def query_for_rescue_type(rescue_value: str):
    if rescue_value == "reset":
        docs = shelter.read({})
        return pd.DataFrame.from_records(docs)

    if rescue_value == "water":
        q = {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": {"$regex": "^Intact", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$lte": 156}
        }
    elif rescue_value == "mountain":
        q = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                              "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": {"$regex": "^Intact", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$lte": 156}
        }
    elif rescue_value == "disaster":
        q = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                              "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": {"$regex": "^Intact", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$lte": 156}
        }
    else:
        q = {}

    docs = shelter.read(q)
    return pd.DataFrame.from_records(docs)


@app.callback(
    Output("datatable-id", "data"),
    Input("filter-type", "value")
)
def update_table(filter_value):
    dff = query_for_rescue_type(filter_value)
    if "_id" in dff.columns:
        dff = dff.drop(columns=["_id"])
    return dff.to_dict("records")


In [43]:
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def highlight_selected_columns(selected_columns):
    selected_columns = selected_columns or []
    return [{"if": {"column_id": c}, "background_color": "#D2F3FF"} for c in selected_columns]


In [44]:
@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
    Input("datatable-id", "derived_virtual_selected_rows")
)
def update_map(viewData, selected_rows):
    if viewData and len(viewData):
        dff = pd.DataFrame(viewData)
    else:
        dff = df_master.copy()

    if dff.empty:
        return [dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48], zoom=10,
            children=[dl.TileLayer(id="base-layer-id")]
        )]

    if not selected_rows:
        selected_rows = [0]
    row = max(0, min(selected_rows[0], len(dff) - 1))

    def pick(colname, idx, coerce_float=False, default=None):
        try:
            val = dff.iloc[row][colname] if colname in dff.columns else dff.iloc[row, idx]
            return float(val) if coerce_float else val
        except Exception:
            return default

    lat = pick("location_lat", 13, coerce_float=True, default=30.75)
    lon = pick("location_long", 14, coerce_float=True, default=-97.48)
    breed = str(pick("breed", 4, default="Unknown"))
    name = str(pick("name", 9, default="Unnamed"))

    leaflet = dl.Map(
        style={"width": "1000px", "height": "500px"},
        center=[lat, lon], zoom=10,
        children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon],
                children=[
                    dl.Tooltip(breed),
                    dl.Popup([html.H1("Animal Name"), html.P(name)]),
                ],
            ),
        ],
    )
    return [leaflet]


In [45]:
@app.callback(
    Output("second-chart-id", "figure"),
    Input("datatable-id", "derived_virtual_data")
)
def update_second_chart(viewData):
    import plotly.express as px
    import pandas as pd

    # Use the table’s current view or fall back to master data
    if viewData and len(viewData):
        dff = pd.DataFrame(viewData)
    else:
        dff = df_master.copy()

    # If breed column is missing entirely, show a helpful empty chart
    if dff.empty or ("breed" not in dff.columns):
        return px.bar(title="No data available (missing 'breed' column)")

    # Build a clean (breed, count) table with clear column names
    top = (
        dff["breed"].fillna("Unknown")
        .value_counts()
        .head(15)
        .reset_index(name="count")        # <-- count column explicitly named here
        .rename(columns={"index": "breed"})  # <-- index becomes 'breed'
    )

    fig = px.bar(top, x="breed", y="count", title="Count of Dogs by Breed (Top 15)")
    fig.update_layout(
        xaxis_title="Breed",
        yaxis_title="Count",
        margin=dict(l=20, r=20, t=50, b=80),
        xaxis_tickangle=-40,  # optional: tilt labels for readability
    )
    return fig


In [46]:
import logging
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)
try:
    app.logger.disabled = True
except Exception:
    pass

app.run_server(mode="inline", debug=False)
